# Scraping database review

This notebook browses `scraping.db`, the scraping module's only implemented database. The top-level `src/storage/` module is still a skeleton, so it has no schema or data to review.

> **Kernel note:** select the project's `.venv` Python interpreter/kernel in VS Code so `pandas` and `src.scraping` imports resolve.

In [1]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Find the repository regardless of the notebook launch directory."""
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'config.yaml').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DB_PATH = REPO_ROOT / 'scraping.db'
print(f'Repository: {REPO_ROOT}')
print(f'Database:   {DB_PATH} ({DB_PATH.stat().st_size if DB_PATH.exists() else 0:,} bytes)')

Repository: /Users/kumo/programming/competitor_product_search
Database:   /Users/kumo/programming/competitor_product_search/scraping.db (15,388,672 bytes)


In [2]:
from src.scraping.storage import ScrapeDB

db = ScrapeDB(DB_PATH)
db.init_db()  # Idempotent: creates the six tables on a fresh, empty database.

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    db.conn,
)
table_summary = pd.DataFrame(
    [
        {'table': name, 'rows': db.conn.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0]}
        for name in tables['name']
    ]
)
display(table_summary)

,table,rows
0,escalations,3
1,golden_samples,16
2,invalid_target_phrases,0
3,parsers,2
4,results,20
5,scrape_runs,18


## `scrape_runs`

The operational log: URL, scraper path, outcome, latency, cost, and the parser used when applicable.

In [3]:
runs = pd.read_sql_query('SELECT * FROM scrape_runs ORDER BY id DESC', db.conn)
display(runs)

runs_with_parser = pd.read_sql_query(
    """
    SELECT r.*, p.site
     AS parser_site, p.version AS parser_version
    FROM scrape_runs AS r
    LEFT JOIN parsers AS p ON p.id = r.winning_parser_id
    ORDER BY r.id DESC
    """,
    db.conn,
)
# display(runs_with_parser)


,id,url,host,site,scraper,scraped_at,outcome,path,winning_parser_id,attempts,model_used,latency_ms,cost,signature,error
0,18,https://www.tesco.com/shop/en-GB/products/312841117,www.tesco.com,tesco,TescoScraper,2026-08-10T22:07:26.048Z,success,fast,3.0,1,None,39500,None,NaN,NaN
1,17,https://www.tesco.com/shop/en-GB/products/276797313,www.tesco.com,tesco,TescoScraper,2026-08-10T21:46:35.005Z,success,fast,3.0,1,None,18411,None,NaN,NaN
2,16,https://www.argos.co.uk/product/8648536,www.argos.co.uk,argos,ArgosScraper,2026-08-08T13:58:32.863Z,success,fast,5.0,1,None,49817,None,NaN,NaN
3,15,https://www.argos.co.uk/product/1153750,www.argos.co.uk,argos,ArgosScraper,2026-08-08T13:57:00.849Z,success,fast,5.0,1,None,42045,None,NaN,NaN
4,14,https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y,www.amazon.co.uk,amazon,AmazonUKScraper,2026-08-08T13:52:08.216Z,success,fast,NaN,1,None,12736,None,NaN,NaN
5,13,https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y,www.amazon.co.uk,amazon,AmazonUKScraper,2026-08-08T13:49:54.089Z,escalated,escalated,NaN,1,None,4129,None,amazon|api_fetch|,"[Errno 8] nodename nor servname provided, or not known"
6,12,https://www.tesco.com/shop/en-GB/products/325456932,www.tesco.com,tesco,TescoScraper,2026-08-08T02:05:34.536Z,success,fast,3.0,1,None,75455,None,NaN,NaN
7,11,https://www.tesco.com/shop/en-GB/clothing/products/323682395,www.tesco.com,tesco,TescoScraper,2026-08-08T02:02:49.562Z,success,fast,3.0,1,None,4427,None,NaN,NaN
8,10,https://www.tesco.com/shop/en-GB/products/325636878,www.tesco.com,tesco,TescoScraper,2026-08-08T02:01:23.263Z,success,fast,3.0,1,None,26776,None,NaN,NaN
9,9,https://www.argos.co.uk/product/1153750,www.argos.co.uk,argos,ArgosScraper,2026-08-08T02:00:04.546Z,success,fast,5.0,1,None,16021,None,NaN,NaN


## `results`

This is the main scraped-product data table. The overview keeps JSON compact; the second view expands `product_data` into its individual ProductData fields.

In [4]:
def shorten(value: object, limit: int = 180) -> object:
    if value is None or pd.isna(value):
        return value
    text = str(value)
    return text if len(text) <= limit else text[:limit] + ' …'


def decode_json(value: object) -> dict:
    if not value:
        return {}
    try:
        decoded = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return {'_unparseable_product_data': value}
    return decoded if isinstance(decoded, dict) else {'_json_value': decoded}


results = pd.read_sql_query('SELECT * FROM results ORDER BY id DESC', db.conn)
results_overview = results.copy()
if 'product_data' in results_overview:
    results_overview['product_data'] = results_overview['product_data'].map(shorten)
display(results_overview)

result_metadata = results.reindex(columns=['id', 'url', 'site', 'scraped_at'])
result_fields = pd.json_normalize(results['product_data'].map(decode_json).tolist())
# results_flat = result_metadata.join(result_fields)
# display(result_fields)

,id,url,site,scraped_at,product_data
0,20,https://www.tesco.com/shop/en-GB/products/312841117,tesco,2026-08-10T22:07:26.022487+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/products/312841117"",""website"":""tesco"",""scraped_at"":""2026-08-10T22:07:26.022487Z"",""source_type"":""html"",""parser_versi..."
1,19,https://www.tesco.com/shop/en-GB/products/276797313,tesco,2026-08-10T21:46:34.978738+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/products/276797313"",""website"":""tesco"",""scraped_at"":""2026-08-10T21:46:34.978738Z"",""source_type"":""html"",""parser_versi..."
2,18,https://www.argos.co.uk/product/8648536,argos,2026-08-08T13:58:32.784895+00:00,"{""url"":""https://www.argos.co.uk/product/8648536"",""website"":""argos"",""scraped_at"":""2026-08-08T13:58:32.784895Z"",""source_type"":""html"",""parser_version"":""cs_2026..."
3,17,https://www.argos.co.uk/product/1153750,argos,2026-08-08T13:57:00.766256+00:00,"{""url"":""https://www.argos.co.uk/product/1153750"",""website"":""argos"",""scraped_at"":""2026-08-08T13:57:00.766256Z"",""source_type"":""html"",""parser_version"":""cs_2026..."
4,16,https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y?th=1&psc=1&language=en_GB&currency=GBP,amazon,2026-08-08T13:55:08.498876+00:00,"{""url"":""https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y?th=1&psc=1&language=en_GB&currency=GBP"",""website"":""amazon"",""scraped_at"":""..."
5,15,https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y?th=1&psc=1&language=en_GB&currency=GBP,amazon,2026-08-08T13:52:08.214263+00:00,"{""url"":""https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y?th=1&psc=1&language=en_GB&currency=GBP"",""website"":""amazon"",""scraped_at"":""..."
6,14,https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y?th=1&psc=1&language=en_GB&currency=GBP,amazon,2026-08-08T13:50:21.397912+00:00,"{""url"":""https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y?th=1&psc=1&language=en_GB&currency=GBP"",""website"":""amazon"",""scraped_at"":""..."
7,13,https://www.tesco.com/shop/en-GB/products/325456932,tesco,2026-08-08T02:05:34.506662+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/products/325456932"",""website"":""tesco"",""scraped_at"":""2026-08-08T02:05:34.506662Z"",""source_type"":""html"",""parser_versi..."
8,12,https://www.tesco.com/shop/en-GB/clothing/products/323682395,tesco,2026-08-08T02:02:49.540433+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/clothing/products/323682395"",""website"":""tesco"",""scraped_at"":""2026-08-08T02:02:49.540433Z"",""source_type"":""html"",""par..."
9,11,https://www.tesco.com/shop/en-GB/products/325636878,tesco,2026-08-08T02:01:23.234817+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/products/325636878"",""website"":""tesco"",""scraped_at"":""2026-08-08T02:01:23.234817Z"",""source_type"":""html"",""parser_versi..."


## `parsers`

Generated parser code is intentionally shortened here. Use `show_blob` below to inspect a full parser.

In [11]:
parsers = pd.read_sql_query('SELECT * FROM parsers ORDER BY id DESC', db.conn)
parsers_overview = parsers.copy()
if 'code' in parsers_overview:
    parsers_overview['code'] = parsers_overview['code'].map(shorten)
display(parsers_overview)

,id,site,version,code,page_type_scope,status,created_at,created_by
0,5,argos,cs_20260807_194728,"from bs4 import BeautifulSoup\nimport re\nimport json\n\ndef parse(html: str, url: str) -> dict:\n # Non-product URL check\n if re.search(r'/(browse|c...",None,active,2026-08-07T19:47:28.845Z,initial
1,3,tesco,cs_20260806_015159,"from bs4 import BeautifulSoup\nimport re\nimport json\n\ndef parse(html: str, url: str) -> dict:\n if re.search(r'/(browse|category|search)(/|$)|/c:\d', ...",None,active,2026-08-06T01:51:59.357Z,initial


## `golden_samples`

Golden snapshots validate newly repaired parsers. The flattened view makes expected ProductData fields easy to compare.

In [12]:
goldens = pd.read_sql_query('SELECT * FROM golden_samples ORDER BY id DESC', db.conn)
goldens_overview = goldens.copy()
for column in ['html_snapshot', 'expected_output']:
    if column in goldens_overview:
        goldens_overview[column] = goldens_overview[column].map(shorten)
display(goldens_overview)

golden_metadata = goldens.reindex(columns=['id', 'site', 'page_type', 'captured_at', 'is_stale'])
golden_fields = pd.json_normalize(goldens['expected_output'].map(decode_json).tolist())
goldens_flat = golden_metadata.join(golden_fields)
display(goldens_flat)

,id,site,page_type,html_snapshot,expected_output,captured_at,is_stale,created_by
0,29,argos,standard,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light""><head><script src=""https://www.paypal.com/sdk/js?client-id=AfpMAdELR-HyL-rxVf_QkvwjvLX...","{""url"": ""https://www.argos.co.uk/product/tuc148159202"", ""website"": ""argos"", ""scraped_at"": ""2026-08-07T19:46:05.016531Z"", ""source_type"": ""html"", ""parser_vers...",2026-08-07T19:47:28.905Z,0,coldstart
1,28,argos,discounted,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script type=""text/javascript"" async="""" src...","{""url"": ""https://www.argos.co.uk/product/7662881"", ""website"": ""argos"", ""scraped_at"": ""2026-08-07T19:46:04.551986Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-07T19:47:28.900Z,0,coldstart
2,27,argos,discounted,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script src=""https://www.paypal.com/sdk/js?...","{""url"": ""https://www.argos.co.uk/product/8695390"", ""website"": ""argos"", ""scraped_at"": ""2026-08-07T19:46:04.110407Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-07T19:47:28.894Z,0,coldstart
3,26,argos,standard,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script type=""text/javascript"" async="""" src...","{""url"": ""https://www.argos.co.uk/product/8463948"", ""website"": ""argos"", ""scraped_at"": ""2026-08-07T19:46:03.678623Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-07T19:47:28.869Z,0,coldstart
4,25,argos,standard,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script src=""https://www.paypal.com/sdk/js?...","{""url"": ""https://www.argos.co.uk/product/7726851"", ""website"": ""argos"", ""scraped_at"": ""2026-08-07T19:46:03.227919Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-07T19:47:28.862Z,0,coldstart
5,24,argos,discounted,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script src=""https://www.paypal.com/sdk/js?...","{""url"": ""https://www.argos.co.uk/product/7831935"", ""website"": ""argos"", ""scraped_at"": ""2026-08-07T19:46:02.768843Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-07T19:47:28.850Z,0,coldstart
6,23,tesco,standard,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.08.05-3c013d45"",""mfe-basket-manager"":""2026.07.29-187409bf"",""mfe-analytics"":""2026.08.05-f4b7f337"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/325340282"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-06T19:59:47.549823Z"", ""source_type"": ""html"", ""pars...",2026-08-06T19:59:47.578Z,0,auto
7,22,tesco,membership,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.08.05-3c013d45"",""mfe-global-scripts"":""2026.08.05-c90a6c66"",""mfe-basket-manager"":""2026.07.29-187409b...","{""url"": ""https://www.tesco.com/shop/en-GB/products/325837129"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-06T19:57:45.421888Z"", ""source_type"": ""html"", ""pars...",2026-08-06T19:57:45.454Z,0,auto
8,21,tesco,discounted,"<!doctype html>\n<html lang=""en-GB""><head>\n <meta charset=""UTF-8"">\n <meta name=""viewport"" content=""width=device-width, initial-scale=1.0"">\n <met...","{""url"": ""https://www.tesco.com/shop/en-GB/products/325355040"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-06T19:51:09.668145Z"", ""source_type"": ""html"", ""pars...",2026-08-06T19:51:09.702Z,0,auto
9,14,tesco,membership,"<!doctype html>\n<html lang=""en-GB""><head>\n <meta charset=""UTF-8"">\n <meta name=""viewport"" content=""width=device-width, initial-scale=1.0"">\n <met...","{""url"": ""https://www.tesco.com/shop/en-GB/products/311961072"", ""web

,id,site,page_type,captured_at,is_stale,url,website,scraped_at,source_type,parser_version,title,brand,gtin,image_urls,variant,price,currency,list_price,membership_price,in_stock,availability_raw,raw,variant.color,variant.size
0,29,argos,standard,2026-08-07T19:47:28.905Z,0,https://www.argos.co.uk/product/tuc148159202,argos,2026-08-07T19:46:05.016531Z,html,coldstart_v1,Yellow & Brown Print Midi Co-ord Skirt 16,Tu,NaN,"[https://media.4rgos.it/i/Argos/tuc148159165-Yellow_R_Z001A, https://media.4rgos.it/i/Argos/tuc148159165-Yellow_R_Z002A, https://media.4rgos.it/i/Argos/tuc1...",NaN,24.00,GBP,NaN,NaN,True,In stock,None,NaN,NaN
1,28,argos,discounted,2026-08-07T19:47:28.900Z,0,https://www.argos.co.uk/product/7662881,argos,2026-08-07T19:46:04.551986Z,html,coldstart_v1,Dyson Airwrap i.d Multi-Styler + Dryer and Diffuser - Pink,Dyson,5025155117316,"[https://media.4rgos.it/i/Argos/7662881_R_Z001A, https://media.4rgos.it/i/Argos/7662881_R_Z002A, https://media.4rgos.it/i/Argos/7662881_R_Z003A, https://med...",NaN,429.99,GBP,479.99,NaN,True,In stock,None,Pink,NaN
2,27,argos,discounted,2026-08-07T19:47:28.894Z,0,https://www.argos.co.uk/product/8695390,argos,2026-08-07T19:46:04.110407Z,html,coldstart_v1,Samsung 55 Inch UE55U7000HKXXU Smart 4K UHD HDR Crystal TV,Samsung,8806099010003,"[https://media.4rgos.it/i/Argos/8695390_R_Z001A, https://media.4rgos.it/i/Argos/8695390_R_A001, https://media.4rgos.it/i/Argos/8695390_R_A002, https://media...",NaN,349.00,GBP,369.00,NaN,True,In stock,None,NaN,55 inch
3,26,argos,standard,2026-08-07T19:47:28.869Z,0,https://www.argos.co.uk/product/8463948,argos,2026-08-07T19:46:03.678623Z,html,coldstart_v1,Northern Hex Cast Iron Dumbbells Set - 2 x 4Kg,Northern,5059283400040,"[https://media.4rgos.it/i/Argos/8463948_R_Z001A, https://media.4rgos.it/i/Argos/8463948_R_Z002A, https://media.4rgos.it/i/Argos/8463948_R_Z003A]",NaN,20.00,GBP,NaN,NaN,True,In stock,None,NaN,4kg
4,25,argos,standard,2026-08-07T19:47:28.862Z,0,https://www.argos.co.uk/product/7726851,argos,2026-08-07T19:46:03.227919Z,html,coldstart_v1,PlayStation 5 Digital Edition - Slim Console,PlayStation,711719020714,"[https://media.4rgos.it/i/Argos/7726851_R_Z001A, https://media.4rgos.it/i/Argos/7726851_R_Z002A, https://media.4rgos.it/i/Argos/7726851_R_Z003A, https://med...",NaN,519.99,GBP,NaN,NaN,True,In stock,None,NaN,NaN
5,24,argos,discounted,2026-08-07T19:47:28.850Z,0,https://www.argos.co.uk/product/7831935,argos,2026-08-07T19:46:02.768843Z,html,coldstart_v1,Ninja Thirsti Black Travel Bottle - 530ml,Ninja,0622356293709,"[https://media.4rgos.it/i/Argos/7831935_R_Z001A, https://media.4rgos.it/i/Argos/7831935_R_Z002A, https://media.4rgos.it/i/Argos/7831935_R_Z003A, https://med...",NaN,24.00,GBP,30.00,NaN,True,In stock,None,Black,530ml
6,23,tesco,standard,2026-08-06T19:59:47.578Z,0,https://www.tesco.com/shop/en-GB/products/325340282,tesco,2026-08-06T19:59:47.549823Z,html,cs_20260806_015159,Set of 6 Outdoor Garden Patio Textilene Furniture Chairs in Black,Samuel Alexander,05056589112851,"[https://digitalcontent.api.tesco.com/v2/media/marketplace/45e31f97-2d6f-4bdf-99e7-3a2489a53591/1b9cf26b0c684cb3b7e3e0a9109be5e0_1977662044.jpeg, https://di...",NaN,100.75,GBP,NaN,NaN,True,In stock,None,NaN,NaN
7,22,tesco,membership,2026-08-06T19:57:45.454Z,0,https://www.tesco.com/shop/en-GB/products/325837129,tesco,2026-08-06T19:57:45.421888Z,html,cs_20260806_015159,"Vital Sign Monitor for SpO2, Blood Pressure, Heart Rate, Body Temperature- Fall & SOS Alert- 24/7 Remote Health & Safety Monitoring- 10 Day Battery- Audar E...",AUDAR,05053047010360,"[https://digitalcontent.api.tesco.com/v2/media/marketplace/c6b151a1-0eed-4650-b1c8-fa15160cbacb/5AiLHrPM-K34NiWK2WYkf4DVs_672504055.jpeg, https://digitalcon...",NaN,174.95,GBP,NaN,148.71,True,In stock,None,NaN,NaN
8,21,tesco,discounted,2026-08-06T19:51:09.702Z,0,https://www.tesco.com/shop/en-GB/products/325355040,tesco,2026-08-06T19:51:09.668145Z,html,cs_20260806_015159,Green & Gold DIY Garland Balloon Arch Kit,Unique Party,000111792

## `escalations`

Escalations are deduplicated by signature. The second view uses the app's store API for the currently open queue.

In [13]:
from src.scraping.storage import EscalationStore

escalations = pd.read_sql_query('SELECT * FROM escalations ORDER BY id DESC', db.conn)
escalations_overview = escalations.copy()
if 'snapshot' in escalations_overview:
    escalations_overview['snapshot'] = escalations_overview['snapshot'].map(shorten)
display(escalations_overview)

open_escalations = pd.DataFrame(EscalationStore(db).get_open())
display(open_escalations)

,id,signature,reason,affected_count,snapshot,status,created_at
0,3,amazon|api_fetch|,api_malformed,1,"{""site"": ""amazon"", ""url"": ""https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y"", ""attempts"": [{""scraper"": ""AmazonUKScraper"", ""failed_...",open,2026-08-08T13:49:54.089Z
1,2,amazon|gate_validation|,api_malformed,3,"{""site"": ""amazon"", ""url"": ""https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y"", ""attempts"": [{""scraper"": ""AmazonUKScraper"", ""failed_...",open,2026-08-08T13:27:08.085Z
2,1,argos|api_infra|,api_malformed,2,"{""site"": ""argos"", ""url"": ""https://www.argos.co.uk/product/tuc143428469"", ""attempts"": [{""scraper"": ""ArgosScraper"", ""failed_stage"": ""parser_broken"", ""errors"":...",open,2026-08-06T20:35:21.678Z


,id,signature,reason,affected_count,snapshot,status,created_at
0,3,amazon|api_fetch|,api_malformed,1,"{""site"": ""amazon"", ""url"": ""https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y"", ""attempts"": [{""scraper"": ""AmazonUKScraper"", ""failed_...",open,2026-08-08T13:49:54.089Z
1,2,amazon|gate_validation|,api_malformed,3,"{""site"": ""amazon"", ""url"": ""https://www.amazon.co.uk/LIVIVO-Heated-Electric-Over-Blanket/dp/B0772VMN9Y"", ""attempts"": [{""scraper"": ""AmazonUKScraper"", ""failed_...",open,2026-08-08T13:27:08.085Z
2,1,argos|api_infra|,api_malformed,2,"{""site"": ""argos"", ""url"": ""https://www.argos.co.uk/product/tuc143428469"", ""attempts"": [{""scraper"": ""ArgosScraper"", ""failed_stage"": ""parser_broken"", ""errors"":...",open,2026-08-06T20:35:21.678Z


## `invalid_target_phrases`

Small lookup table used to recognize pages that are not product targets.

In [14]:
phrases = pd.read_sql_query('SELECT * FROM invalid_target_phrases ORDER BY id DESC', db.conn)
display(phrases)

,id,site,phrase,source,added_at


## Full-field drill-down

Overview tables truncate long code and snapshots. Call `show_blob` with a table, row id, and column to print the complete value; JSON is formatted for readability.

In [6]:
REVIEW_TABLES = {
    'parsers', 'golden_samples', 'scrape_runs', 'results', 'escalations', 'invalid_target_phrases'
}


def show_blob(table: str, row_id: int, column: str) -> None:
    """Print one complete text/JSON field from a reviewed table."""
    if table not in REVIEW_TABLES:
        raise ValueError(f'Unknown review table: {table}')
    valid_columns = {row['name'] for row in db.conn.execute(f'PRAGMA table_info({table})')}
    if column not in valid_columns:
        raise ValueError(f'Unknown column for {table}: {column}')

    row = db.conn.execute(
        f'SELECT {column} FROM {table} WHERE id = ?', (row_id,)
    ).fetchone()
    if row is None:
        raise LookupError(f'No {table} row with id={row_id}')

    value = row[0]
    if value is None:
        print('(NULL)')
        return
    try:
        print(json.dumps(json.loads(value), indent=2, ensure_ascii=False, default=str))
    except (TypeError, json.JSONDecodeError):
        print(value)


# Examples:
# show_blob('parsers', 1, 'code')
# show_blob('golden_samples', 1, 'html_snapshot')
show_blob('results', 1, 'product_data')
# show_blob('escalations', 1, 'snapshot')

{
  "url": "https://www.tesco.com/shop/en-GB/products/325355040",
  "website": "tesco",
  "scraped_at": "2026-08-06T19:51:09.668145Z",
  "source_type": "html",
  "parser_version": "cs_20260806_015159",
  "title": "Green & Gold DIY Garland Balloon Arch Kit",
  "brand": "Unique Party",
  "gtin": "00011179282012",
  "image_urls": [
    "https://digitalcontent.api.tesco.com/v2/media/marketplace/0fcddccf-55a9-40a7-abf3-e6d2d6bc2331/UcxF0G9LUZNViIKb5HLwVwbDw_2082540610.jpeg",
    "https://digitalcontent.api.tesco.com/v2/media/marketplace/aad6ac24-c3a5-4aa6-be1c-5221cccab41d/2ef79a8e0d344b7097b8d70dbb61e649_1328819889.jpeg"
  ],
  "variant": null,
  "price": "4.29",
  "currency": "GBP",
  "list_price": "7.29",
  "membership_price": null,
  "in_stock": true,
  "availability_raw": "In stock",
  "raw": null
}


## Handy filtered questions

For common site-level questions, the purpose-built stores are more convenient than writing the aggregation again. Change `SITE` and re-run this cell.

In [16]:
from src.scraping.storage import ParserStore, RunStore

SITE = 'tesco'
active_parsers = pd.DataFrame(ParserStore(db).get_active_ordered_by_hits(SITE))
parser_hit_rates = pd.DataFrame(RunStore(db).get_hit_rates(SITE))

print(f'Active parsers for {SITE}:')
display(active_parsers)
print(f'Parser hit rates for {SITE}:')
display(parser_hit_rates)

Active parsers for tesco:


,id,site,version,code,page_type_scope,status,created_at,created_by,hits
0,3,tesco,cs_20260806_015159,"from bs4 import BeautifulSoup\nimport re\nimport json\n\ndef parse(html: str, url: str) -> dict:\n if re.search(r'/(browse|category|search)(/|$)|/c:\d', ...",None,active,2026-08-06T01:51:59.357Z,initial,7


Parser hit rates for tesco:


,winning_parser_id,hits
0,3,7


## 清空数据库 (Clear database by site)

⛔  **DESTRUCTIVE**. Permanently deletes the site's rows from `parsers` and
`golden_samples`.  Also sets `scrape_runs.winning_parser_id = NULL` where needed
to satisfy the foreign-key constraint (the run history rows are kept).

Does **not** touch `results`, `invalid_target_phrases`, or `escalations`.

This is typically used before re-cold-starting a site.
Run the preview cell below first to see what data exists for each site.

In [3]:
# Preview site data — all site-scoped tables for context.
# A clear will delete parsers + golden_samples rows; scrape_runs
# FK pointers are NULLed but the runs themselves are kept.
all_site_tables = ['scrape_runs', 'results', 'invalid_target_phrases', 'parsers', 'golden_samples']
clear_targets = {'parsers', 'golden_samples'}

sites: set[str] = set()
for t in all_site_tables:
    for r in db.conn.execute(f"SELECT DISTINCT site FROM {t}"):
        sites.add(r["site"])

if sites:
    print("Sites in database (canonical keys) and row counts:")
    for site in sorted(sites):
        parts = []
        clear_total = 0
        for t in all_site_tables:
            n = db.conn.execute(
                f"SELECT COUNT(*) FROM {t} WHERE site = ?", (site,)
            ).fetchone()[0]
            if n:
                marker = " ← WILL BE CLEARED" if t in clear_targets else ""
                parts.append(f"{t}={n}{marker}")
                if t in clear_targets:
                    clear_total += n
        print(f"  {site}: {clear_total} rows to clear ({', '.join(parts)})"
              if parts else f"  {site}: (empty)")
else:
    print("No site-scoped data found.")

Sites in database (canonical keys) and row counts:
  tesco: 6 rows to clear (parsers=1 ← WILL BE CLEARED, golden_samples=5 ← WILL BE CLEARED)


In [ ]:
# Clear a site — set the values below and run this cell
SITE = ""           # ← canonical site key, e.g. 'tesco'
CONFIRM = False     # ← set to True to execute the hard delete

if SITE and CONFIRM:
    counts = db.clear_site(SITE)
    detail = ", ".join(
        f"{k}={v}" for k, v in counts.items() if v and k != "scrape_runs_detached"
    )
    detached = counts.get("scrape_runs_detached", 0)
    if detached:
        detail += f" ({detached} scrape_runs FK detached)"
    print(f"✓ Site '{SITE}' cleared: {detail}")
else:
    print("Set SITE and CONFIRM = True to clear a site.")

✓ Site 'argos' cleared: parsers=1, golden_samples=6
